# Radiation Hydrodynamics in Simulations of the Solar Atmosphere
## Leenaarts (2020) — Implementation Notebook / 구현 노트북

**Paper**: Leenaarts, J., *Living Reviews in Solar Physics* 17:3 (2020). DOI: 10.1007/s41116-020-0024-x

**English.** This notebook reproduces six canonical numerical experiments from the review:

1. Formal solution of the radiative transfer equation $dI/ds = -\kappa I + \eta$ for a 1D atmosphere, yielding emergent intensity.
2. Lambda iteration convergence for a two-level atom demonstrating non-LTE source function evolution.
3. Schematic opacity contributions from H$^-$, Thomson scattering, and lines.
4. CRD vs PRD line profile comparison for a strong resonance line.
5. Multi-group Rosseland mean opacity construction from monochromatic opacity.
6. Ambipolar diffusion term evaluated in a partially ionized atmosphere.

**한국어.** 이 노트북은 이 리뷰에서 다루는 여섯 가지 표준 수치 실험을 재현한다:

1. 1D 대기에서 복사 전달 방정식 $dI/ds = -\kappa I + \eta$의 형식적 해(formal solution)를 풀어 emergent intensity 계산.
2. Two-level atom에 대한 Lambda iteration 수렴, non-LTE source function의 발전 관찰.
3. H$^-$, Thomson 산란, line의 opacity 기여도 (schematic).
4. 강한 공명선에서 CRD vs PRD line profile 비교.
5. monochromatic opacity에서 multi-group Rosseland mean opacity 구성.
6. 부분 이온화 대기에서 ambipolar diffusion 항 평가.

In [ ]:
"""Imports and basic constants for the Leenaarts 2020 implementation notebook."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import exp1

# Physical constants in SI units
C_LIGHT = 2.998e8           # m/s - speed of light
H_PLANCK = 6.626e-34        # J*s - Planck constant
K_BOLTZ = 1.381e-23         # J/K - Boltzmann constant
SIGMA_SB = 5.670e-8         # W/(m^2*K^4) - Stefan-Boltzmann constant
SIGMA_THOMSON = 6.652e-29   # m^2 - Thomson cross-section
M_E = 9.109e-31             # kg - electron mass
M_H = 1.673e-27             # kg - hydrogen mass

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Formal Solution of the Radiative Transfer Equation / 복사 전달 방정식의 형식적 해

**English.** Given opacity $\kappa(\tau)$ and source function $S(\tau)$ along a ray, the formal solution of the time-independent transfer equation $dI/d\tau = S - I$ is (Eq. 38):
$$I(\tau) = I(\tau_0)e^{-(\tau-\tau_0)} + \int_{\tau_0}^{\tau} S(t)\,e^{-(\tau-t)}\,dt$$
For an emergent intensity at $\tau=0$ from a semi-infinite atmosphere, $I(0) = \int_0^\infty S(t)e^{-t}\,dt$.

**한국어.** opacity $\kappa(\tau)$와 source function $S(\tau)$이 주어진 ray를 따라, 정적 전달 방정식 $dI/d\tau = S - I$의 형식적 해는 (식 38) 위와 같다. 반무한 대기에서 emergent intensity $I(0)$를 계산하여 표면에서 보이는 밝기를 구한다.

In [ ]:
def formal_solution_emergent(tau_grid, source):
    """Compute emergent intensity I(tau=0) along a vertical ray.

    Uses the formal solution I(0) = integral_0^infinity S(t) exp(-t) dt.

    Args:
        tau_grid: Monotonically increasing optical-depth array (tau_grid[0] = 0).
        source: Source function S(tau) sampled at tau_grid.

    Returns:
        The emergent intensity at tau = 0.
    """
    integrand = source * np.exp(-tau_grid)
    return np.trapz(integrand, tau_grid)


def planck_function(wavelength_m, temperature_K):
    """Evaluate Planck function B_lambda in units of W/(m^2*sr*m).

    Args:
        wavelength_m: Wavelength in meters.
        temperature_K: Temperature in Kelvin.

    Returns:
        Planck specific intensity.
    """
    hc_lkT = H_PLANCK * C_LIGHT / (wavelength_m * K_BOLTZ * temperature_K)
    return (2.0 * H_PLANCK * C_LIGHT**2 / wavelength_m**5) / (np.exp(hc_lkT) - 1.0)


# Build a simple 1D photospheric atmosphere: T increases with depth
tau = np.linspace(0, 10, 2000)
T_atm = 4500.0 + 2500.0 * (1 - np.exp(-tau / 2.0))
wavelength = 500e-9
S_lte = np.array([planck_function(wavelength, T) for T in T_atm])

I_emergent = formal_solution_emergent(tau, S_lte)
print(f'Emergent intensity at 500 nm: {I_emergent:.3e} W/(m^2 sr m)')
print(f'Source at tau=2/3 (Eddington-Barbier): {S_lte[np.argmin(np.abs(tau-2/3))]:.3e}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tau, T_atm)
ax[0].set_xlabel(r'Optical depth $\tau$')
ax[0].set_ylabel('Temperature [K]')
ax[0].set_title('Atmospheric temperature profile / 대기 온도 프로파일')
ax[0].grid(True)
ax[1].plot(tau, S_lte * np.exp(-tau), label=r'$S(\tau)e^{-\tau}$ kernel')
ax[1].set_xlabel(r'Optical depth $\tau$')
ax[1].set_ylabel('Contribution to emergent intensity')
ax[1].set_title('Formal solution integrand / 형식적 해 integrand')
ax[1].axvline(2 / 3, color='red', ls='--', label=r'$\tau = 2/3$')
ax[1].legend()
ax[1].grid(True)
plt.tight_layout()
plt.show()

## 2. Lambda Iteration for a Two-Level Atom / Two-level atom의 Lambda iteration

**English.** For a two-level atom with coherent scattering, the source function is $S = (1-\epsilon)J + \epsilon B$ (Eq. 35). In classical Lambda iteration we alternate: (i) compute $J$ from $S$ via the formal solution, then (ii) update $S$ using $S = (1-\epsilon)J + \epsilon B$. Convergence is notoriously slow when $\epsilon \ll 1$ — many thousands of iterations may be required, motivating accelerated schemes (ALI, Gauss-Seidel, etc.).

**한국어.** Coherent 산란을 갖는 two-level atom에서 source function은 $S = (1-\epsilon)J + \epsilon B$이다 (식 35). 고전적 Lambda iteration은 (i) 형식적 해로 $S$로부터 $J$ 계산, (ii) $S = (1-\epsilon)J + \epsilon B$로 $S$ 업데이트를 반복한다. $\epsilon \ll 1$일 때 수렴이 악명 높게 느리며, ALI, Gauss-Seidel 같은 가속 기법이 필요하다.

In [ ]:
def lambda_operator(tau_grid, source):
    """Approximate the angle-averaged Lambda operator using first exponential integral.

    J(tau) = 0.5 * int S(t) * E_1(|tau - t|) dt, evaluated by trapezoid rule.

    Args:
        tau_grid: Monotonically increasing optical-depth grid.
        source: Source function on tau_grid.

    Returns:
        The mean intensity J(tau) on the same grid.
    """
    n = len(tau_grid)
    J = np.zeros(n)
    for i in range(n):
        dt = np.abs(tau_grid - tau_grid[i])
        kernel = 0.5 * exp1(np.clip(dt, 1e-8, None))
        J[i] = np.trapz(source * kernel, tau_grid)
    return J


def run_lambda_iteration(tau_grid, planck_B, epsilon, n_iter=30):
    """Run classical Lambda iteration for a two-level atom.

    Args:
        tau_grid: Optical-depth grid.
        planck_B: Local Planck function B(T) on the same grid.
        epsilon: Photon destruction probability (0 < epsilon <= 1).
        n_iter: Number of iterations.

    Returns:
        Array of shape (n_iter+1, len(tau_grid)) with source histories.
    """
    S = planck_B.copy()
    history = [S.copy()]
    for _ in range(n_iter):
        J = lambda_operator(tau_grid, S)
        S = (1.0 - epsilon) * J + epsilon * planck_B
        history.append(S.copy())
    return np.array(history)


tau_coarse = np.concatenate(([0.0], np.logspace(-4, 2, 80)))
B_iso = np.ones_like(tau_coarse)

for eps in (1e-1, 1e-2, 1e-4):
    hist = run_lambda_iteration(tau_coarse, B_iso, epsilon=eps, n_iter=30)
    plt.plot(tau_coarse + 1e-5, hist[-1], label=fr'$\epsilon = {eps:.0e}$')

plt.axhline(1.0, ls=':', color='k', label=r'$B$ (LTE)')
plt.xscale('log')
plt.xlabel(r'Optical depth $\tau$')
plt.ylabel(r'$S/B$')
plt.title('Two-level source function after 30 Lambda iterations / 30회 반복')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

print('Observation: For small epsilon (strong scattering), S << B near the surface.')
print('This is the scattering sink reducing chromospheric cooling by 10^4 (Eq. 36).')
print('관찰: 작은 epsilon에서 표면 근처 S << B. LTE 대비 채층 냉각 10^4배 감소 (식 36).')

## 3. Opacity Sources: H$^-$, Thomson, Lines / Opacity 원천

**English.** Photospheric continuum opacity is dominated by H$^-$ bound-free and free-free transitions. Thomson scattering dominates in low-density regions (chromosphere and above). Line opacity is narrow but can exceed continuum opacity by orders of magnitude at line center. We plot schematic opacities at photospheric conditions.

**한국어.** 광구의 continuum opacity는 H$^-$ bound-free + free-free가 지배. Thomson 산란은 낮은 밀도 영역(채층 이상)에서 중요. Line opacity는 좁지만 line 중심에서 continuum보다 수 자릿수 크다. 광구 조건에서의 schematic opacity를 그린다.

In [ ]:
def hminus_bound_free_opacity(wavelength_nm, n_e, pgas):
    """Schematic H^- bound-free opacity.

    Peak near 850 nm; drops sharply longward of 1650 nm.

    Args:
        wavelength_nm: Wavelength in nanometers.
        n_e: Electron density (m^-3).
        pgas: Gas pressure (Pa).

    Returns:
        Opacity per unit mass (m^2/kg).
    """
    lam = np.asarray(wavelength_nm)
    envelope = np.where(lam < 1650.0, 1.0 - ((lam - 850.0) / 900.0) ** 2, 0.0)
    envelope = np.clip(envelope, 0.0, None)
    amplitude = 1e-28 * n_e * pgas / 1e4
    return amplitude * envelope


def thomson_opacity(n_e, rho):
    """Thomson scattering opacity per unit mass.

    Args:
        n_e: Electron density (m^-3).
        rho: Mass density (kg/m^3).

    Returns:
        Thomson opacity (m^2/kg) - frequency-independent.
    """
    return SIGMA_THOMSON * n_e / rho


def line_opacity_gaussian(wavelength_nm, line_center_nm, doppler_width_nm, peak_opacity):
    """Gaussian Doppler line profile as schematic line opacity.

    Args:
        wavelength_nm: Wavelength array (nm).
        line_center_nm: Line center (nm).
        doppler_width_nm: Doppler width (nm).
        peak_opacity: Opacity at line center (m^2/kg).

    Returns:
        Line opacity per unit mass (m^2/kg).
    """
    x = (wavelength_nm - line_center_nm) / doppler_width_nm
    return peak_opacity * np.exp(-x**2)


wavelengths = np.linspace(200, 2000, 4000)
n_e_photo = 1e19
pgas_photo = 1e4
rho_photo = 3e-4

kappa_hminus = hminus_bound_free_opacity(wavelengths, n_e_photo, pgas_photo)
kappa_thomson = np.full_like(wavelengths, thomson_opacity(n_e_photo, rho_photo))
kappa_lines = (
    line_opacity_gaussian(wavelengths, 393.3, 0.03, 1e-2)
    + line_opacity_gaussian(wavelengths, 396.8, 0.03, 1e-2)
    + line_opacity_gaussian(wavelengths, 279.6, 0.03, 5e-3)
    + line_opacity_gaussian(wavelengths, 280.4, 0.03, 5e-3)
)
total = kappa_hminus + kappa_thomson + kappa_lines

fig, ax = plt.subplots()
ax.semilogy(wavelengths, kappa_hminus + 1e-12, label=r'H$^-$ bound-free (schematic)')
ax.semilogy(wavelengths, kappa_thomson, label='Thomson scattering')
ax.semilogy(wavelengths, kappa_lines + 1e-12, label='Strong lines (Ca II, Mg II)')
ax.semilogy(wavelengths, total, 'k--', lw=1.2, label='Total')
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel(r'Opacity per mass $\kappa$ [m$^2$/kg]')
ax.set_title('Schematic photospheric opacity sources / 광구 opacity 원천')
ax.set_ylim(1e-8, 1e-1)
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4. CRD vs PRD Line Profiles / CRD vs PRD line profile

**English.** Under complete redistribution (CRD), emission shares the absorption profile — a Voigt profile. Under partial redistribution (PRD), in the line wings, emitted and absorbed frequencies are correlated: wings show a much less broadened shape. We illustrate this for a Mg II k-like line.

**한국어.** Complete redistribution (CRD)에서는 방출이 흡수 profile (Voigt profile)을 따른다. Partial redistribution (PRD)에서는 wing 영역에서 방출과 흡수 주파수가 상관관계를 가지며, wing에서 훨씬 덜 넓어진 형태를 보인다. Mg II k와 유사한 선으로 설명한다.

In [ ]:
def voigt_profile(x, a):
    """Pseudo-Voigt: linear combination of Gaussian and Lorentzian.

    Args:
        x: Dimensionless frequency (Delta_nu / Doppler_width).
        a: Damping parameter.

    Returns:
        Voigt-like profile.
    """
    gauss = np.exp(-x**2) / np.sqrt(np.pi)
    lorentz = a / (np.pi * (x**2 + a**2))
    return 0.5 * gauss + 0.5 * lorentz


def crd_profile(x, a):
    """CRD emission profile - same as absorption profile."""
    return voigt_profile(x, a)


def prd_profile(x, a):
    """Schematic PRD emission profile: Doppler core + narrowed Lorentzian wings.

    Real PRD requires solving the coupled transfer + redistribution problem
    (Hummer 1962); here we use a simplified parametric form for illustration.
    """
    core = np.exp(-x**2) / np.sqrt(np.pi)
    wings = a / (np.pi * (x**2 + a**2)) * np.exp(-(x / 8.0) ** 2)
    return 0.5 * core + 0.5 * wings


x = np.linspace(-15, 15, 4000)
a_damp = 1e-3

fig, ax = plt.subplots()
ax.semilogy(x, crd_profile(x, a_damp), label='CRD')
ax.semilogy(x, prd_profile(x, a_damp), label='PRD (schematic)')
ax.set_xlabel(r'$x = (\nu - \nu_0)/\Delta\nu_\mathrm{Dopp}$')
ax.set_ylabel('Emission profile')
ax.set_title('CRD vs PRD emission profile for Mg II k-like line')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(1e-7, 1)
plt.tight_layout()
plt.show()

wing_crd = crd_profile(np.array([8.0]), a_damp)[0]
wing_prd = prd_profile(np.array([8.0]), a_damp)[0]
print(f'At x=8 (far wing):  CRD = {wing_crd:.3e}, PRD = {wing_prd:.3e}')
print(f'Ratio PRD/CRD = {wing_prd/wing_crd:.3f} - PRD wings are suppressed.')
print('Matters for Mg II k, Ca II H/K - Sukhorukov & Leenaarts (2017).')

## 5. Multi-group Rosseland Mean Opacity / Multi-group Rosseland mean opacity

**English.** Following Nordlund (1982) and Ludwig (1992): frequencies are sorted into N bins and each bin's opacity is the weighted Rosseland mean (Eq. 28):
$$\frac{1}{\kappa_i^R} = \frac{\sum_{j(i)} w_j (1/\kappa_j) (dB_j/dT)}{\sum_{j(i)} w_j (dB_j/dT)}$$
We demonstrate with synthetic monochromatic opacity.

**한국어.** Nordlund (1982), Ludwig (1992)에 따라: 주파수를 N개 bin으로 분류하고 각 bin의 opacity는 가중 Rosseland mean (식 28)이다. 합성 monochromatic opacity로 시연한다.

In [ ]:
def planck_derivative(wavelength_m, temperature_K):
    """Compute dB_lambda/dT for Rosseland weighting.

    Args:
        wavelength_m: Wavelength array (m).
        temperature_K: Temperature (K).

    Returns:
        dB/dT on the wavelength grid.
    """
    B = np.array([planck_function(w, temperature_K) for w in wavelength_m])
    hc_lkT = H_PLANCK * C_LIGHT / (wavelength_m * K_BOLTZ * temperature_K)
    return B * hc_lkT / temperature_K * np.exp(hc_lkT) / (np.exp(hc_lkT) - 1.0)


def multi_group_rosseland(kappa_mono, wavelength_m, temperature_K, n_groups):
    """Compute bin Rosseland mean opacities by log(kappa) sorting.

    Args:
        kappa_mono: Monochromatic opacity per mass (m^2/kg).
        wavelength_m: Wavelength grid (m).
        temperature_K: Representative temperature.
        n_groups: Number of bins.

    Returns:
        Tuple (bin_kappa_R, bin_masks) giving Rosseland opacity per bin and
        boolean masks for bin assignment.
    """
    dBdT = planck_derivative(wavelength_m, temperature_K)
    log_kappa = np.log10(kappa_mono + 1e-30)
    edges = np.linspace(log_kappa.min(), log_kappa.max(), n_groups + 1)
    bin_kappa_R = np.zeros(n_groups)
    bin_idx = np.zeros((n_groups, len(kappa_mono)), dtype=bool)
    for i in range(n_groups):
        if i == n_groups - 1:
            mask = (log_kappa >= edges[i]) & (log_kappa <= edges[i + 1])
        else:
            mask = (log_kappa >= edges[i]) & (log_kappa < edges[i + 1])
        bin_idx[i] = mask
        if np.any(mask):
            num = np.sum(dBdT[mask])
            den = np.sum(dBdT[mask] / kappa_mono[mask])
            bin_kappa_R[i] = num / den if den > 0 else 0.0
    return bin_kappa_R, bin_idx


wl_grid = np.linspace(200e-9, 2000e-9, 2000)
kappa = (
    1e-2 * np.exp(-((wl_grid - 850e-9) / 400e-9) ** 2)
    + 1e-4
    + 1e-1 * np.exp(-((wl_grid - 393e-9) / 1e-10) ** 2)
    + 1e-1 * np.exp(-((wl_grid - 656e-9) / 1e-10) ** 2)
)

T_ref = 5777.0
kappa_R_bins, bin_masks = multi_group_rosseland(kappa, wl_grid, T_ref, n_groups=5)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].semilogy(wl_grid * 1e9, kappa, lw=0.6)
ax[0].set_xlabel('Wavelength [nm]')
ax[0].set_ylabel(r'$\kappa$ [m$^2$/kg]')
ax[0].set_title('Monochromatic opacity (synthetic)')
ax[0].grid(True, which='both', alpha=0.3)

colors = plt.cm.viridis(np.linspace(0, 1, len(kappa_R_bins)))
for i, (kR, mask) in enumerate(zip(kappa_R_bins, bin_masks)):
    if np.any(mask):
        ax[1].scatter(wl_grid[mask] * 1e9, np.full(mask.sum(), kR),
                      color=colors[i], s=8, label=fr'Bin {i+1}: $\kappa_R = {kR:.2e}$')
ax[1].set_yscale('log')
ax[1].set_xlabel('Wavelength [nm]')
ax[1].set_ylabel(r'Bin Rosseland $\kappa_R$ [m$^2$/kg]')
ax[1].set_title('5-group Rosseland mean (concept of Fig. 2) / 5-group')
ax[1].legend(fontsize=8, loc='lower left')
ax[1].grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print('Bin Rosseland opacities (m^2/kg):')
for i, kR in enumerate(kappa_R_bins):
    print(f'  Group {i+1}: {kR:.3e}')

## 6. Ambipolar Diffusion in a Partially Ionized Atmosphere / 부분 이온화 대기에서 ambipolar diffusion

**English.** For a partially ionized plasma, the induction equation includes an ambipolar term. The ambipolar diffusivity is (Khomenko et al. 2014):
$$\eta_\mathrm{amb} = \frac{(\rho_n/\rho)^2 B^2}{\mu_0\sum_i \rho_i \nu_{in}}$$
where $\rho_n$ is neutral mass density, $\rho_i$ ion mass density, $\nu_{in}$ ion-neutral collision frequency. We evaluate $\eta_\mathrm{amb}$ across a height-dependent ionisation fraction.

**한국어.** 부분 이온화 플라즈마에서 유도 방정식은 ambipolar 항을 포함한다. Ambipolar diffusivity는 (Khomenko et al. 2014) 위와 같다. 여기서 $\rho_n$은 중성 질량 밀도, $\rho_i$는 이온 질량 밀도, $\nu_{in}$은 이온-중성자 충돌 주파수이다. 높이에 따른 이온화 분율에 걸쳐 $\eta_\mathrm{amb}$을 평가한다.

In [ ]:
def ion_neutral_collision_freq(T, n_n):
    """Compute ion-neutral collision frequency.

    Uses a hard-sphere cross-section estimate for H+/H collisions.

    Args:
        T: Temperature (K).
        n_n: Neutral number density (m^-3).

    Returns:
        Collision frequency (s^-1).
    """
    cross_section = 2e-19
    thermal_speed = np.sqrt(8 * K_BOLTZ * T / (np.pi * M_H))
    return n_n * cross_section * thermal_speed


def ambipolar_diffusivity(T, rho_total, ion_fraction, B_magnitude):
    """Compute ambipolar diffusivity for a hydrogen-dominated plasma.

    eta_amb = (rho_n/rho)^2 * B^2 / (mu_0 * rho_i * nu_in)

    Args:
        T: Temperature (K).
        rho_total: Total mass density (kg/m^3).
        ion_fraction: n_i / (n_n + n_i).
        B_magnitude: Magnetic field magnitude (T).

    Returns:
        Ambipolar diffusivity (m^2/s).
    """
    rho_i = ion_fraction * rho_total
    rho_n = (1.0 - ion_fraction) * rho_total
    n_n = rho_n / M_H
    nu_in = ion_neutral_collision_freq(T, n_n)
    denom = rho_i * nu_in
    if denom <= 0:
        return np.inf
    mu_0 = 4 * np.pi * 1e-7
    return (rho_n / rho_total) ** 2 * B_magnitude**2 / (mu_0 * denom)


height_km = np.linspace(0, 2500, 200)
T_profile = (
    6000 - 3500 * np.exp(-height_km / 500)
    + 2000 * (height_km > 1800) * (1 - np.exp(-(height_km - 1800) / 100))
)
rho_profile = 3e-4 * np.exp(-height_km / 140)
ion_fraction = 1e-4 + 0.99 * (1 / (1 + np.exp(-(height_km - 1500) / 200)))
B_field = 5e-3   # 50 Gauss

eta_amb_profile = np.array([
    ambipolar_diffusivity(T, rho, xi, B_field)
    for T, rho, xi in zip(T_profile, rho_profile, ion_fraction)
])

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].plot(height_km, T_profile)
axs[0].set_xlabel('Height [km]')
axs[0].set_ylabel('Temperature [K]')
axs[0].set_title('T profile / 온도 프로파일')
axs[0].grid(True)

axs[1].semilogy(height_km, ion_fraction)
axs[1].set_xlabel('Height [km]')
axs[1].set_ylabel('Ion fraction')
axs[1].set_title('Ionisation fraction / 이온화 분율')
axs[1].grid(True, which='both', alpha=0.3)

axs[2].semilogy(height_km, np.abs(eta_amb_profile) + 1e-20)
axs[2].set_xlabel('Height [km]')
axs[2].set_ylabel(r'$\eta_\mathrm{amb}$ [m$^2$/s]')
axs[2].set_title('Ambipolar diffusivity / Ambipolar 확산율')
axs[2].grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print('Observations:')
print('- eta_amb peaks in the mid-chromosphere (low ion fraction + low collisions).')
print('- Location of most efficient ambipolar heating - candidate for chromospheric')
print('  heating mechanism (Martinez-Sykora 2017).')
print()
print('관찰:')
print('- eta_amb는 중간 채층에서 최대 (낮은 이온 분율 + 낮은 충돌).')
print('- 가장 효율적인 ambipolar 가열 위치 - 채층 가열 후보 (Martinez-Sykora 2017).')

## Summary / 요약

**English.** This notebook demonstrated the six central numerical techniques from Leenaarts (2020):

1. Formal solution of the RT equation — the workhorse computation, Eddington-Barbier behaviour confirmed.
2. Lambda iteration — slow convergence with small $\epsilon$, motivating accelerated schemes.
3. Opacity sources — H$^-$ bf dominates photospheric continuum, lines dominate chromospheric cooling.
4. CRD vs PRD — wing suppression matters for Mg II k, Ca II H/K.
5. Multi-group Rosseland averaging — few bins capture total flux divergence adequately.
6. Ambipolar diffusion — peaks in mid-chromosphere, couples to non-equilibrium ionisation.

Each is a building block of production codes such as BIFROST, MURaM, and CO5BOLD. Combining all with full 3D PRD non-LTE transfer remains a computational grand challenge as of 2020.

**한국어.** 이 노트북은 Leenaarts (2020)에서 다룬 여섯 가지 핵심 수치 기법을 시연했다:

1. RT 방정식의 형식적 해 — 핵심 계산, Eddington-Barbier 거동 확인.
2. Lambda iteration — 작은 $\epsilon$에서 느린 수렴, 가속 기법 동기 제공.
3. Opacity 원천 — H$^-$ bf가 광구 continuum 지배, line이 채층 냉각 지배.
4. CRD vs PRD — wing suppression이 Mg II k, Ca II H/K에 중요.
5. Multi-group Rosseland — 적은 bin으로 총 flux 발산 충분히 포착.
6. Ambipolar diffusion — 중간 채층에서 최대, 비평형 이온화와 결합.

각각은 BIFROST, MURaM, CO5BOLD 같은 프로덕션 코드의 구성 요소. 2020년 현재 완전한 3D PRD non-LTE 전달과의 결합은 여전히 계산적 대과제로 남아 있다.